In [3]:
"""
Fasal Nirnay — XGBoost + LSTM Hybrid Price Prediction
Loads directly from /kaggle/input/datasets/mruddunijmodha/merged-engineered

XGBoost : GPU (hist / cuda)
LSTM     : CPU  (cuDNN kernel image incompatibility workaround)
"""

import warnings
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────
INPUT_DIR  = Path("/kaggle/input/datasets/mruddunijmodha/merged-engineered")
OUTPUT_DIR = Path("/kaggle/working/pipeline_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CUTOFF = pd.Timestamp("2022-01-01")

# XGBoost gets GPU; LSTM stays on CPU to avoid cuDNN kernel mismatch
xgb_device  = "cuda" if torch.cuda.is_available() else "cpu"
lstm_device = torch.device("cpu")   # forced — cuDNN RNN kernel not available
print(f"XGBoost device : {xgb_device}")
print(f"LSTM device    : cpu (forced)")

# ── Feature lists ─────────────────────────────────────────────
WEATHER_FEATS = [
    "temperature_mean", "temperature_max", "temperature_min",
    "humidity", "precipitation", "rainfall",
    "wind_speed", "wind_direction",
    "soil_temperature_0_7cm", "soil_moisture_0_7cm", "solar_radiation",
    "temp_range", "heat_stress", "frost_risk", "drought_flag", "vpd", "soil_stress",
]
CALENDAR_FEATS = [
    "year", "month", "quarter", "week_of_year", "day_of_year",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
]
CAT_FEATS = [
    "Commodity_enc", "Variety_enc", "season_enc",
    "weather_condition_enc", "state_norm_enc", "district_norm_enc",
]
LAG_FEATS = [
    "price_lag1", "price_lag7", "price_lag30",
    "price_roll7", "price_roll14", "price_roll30",
    "price_std7", "price_std14", "price_std30",
    "daily_listings",
]
PRICE_FEATS = WEATHER_FEATS + CALENDAR_FEATS + CAT_FEATS + LAG_FEATS
GRADE_FEATS = WEATHER_FEATS + CALENDAR_FEATS + CAT_FEATS

SEQ_FEATS = [
    "price_lag1", "price_lag7", "price_lag30",
    "price_roll7", "price_roll14", "price_roll30",
    "price_std7", "price_std14", "price_std30",
]

# ═══════════════════════════════════════════════════════════════
# 1. LOAD
# ═══════════════════════════════════════════════════════════════
print("\nLoading merged-engineered parquet …")
parquet_files = sorted(INPUT_DIR.rglob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

df = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df):,} rows  |  {len(parquet_files)} file(s)")

# ═══════════════════════════════════════════════════════════════
# 2. PREPARE
# ═══════════════════════════════════════════════════════════════
available_price_feats = [f for f in PRICE_FEATS if f in df.columns]
missing = set(PRICE_FEATS) - set(available_price_feats)
if missing:
    print(f"  [WARN] Missing features (skipped): {missing}")

sub = df.dropna(subset=available_price_feats + ["Min_Price"]).copy()
if len(sub) > 5_000_000:
    sub = sub.sample(5_000_000, random_state=42)
    print(f"  Sampled to 5M rows")

X_all = sub[available_price_feats].values.astype(np.float32)
y_all = sub["Min_Price"].values.astype(np.float32)
mask  = (sub["date"] < TRAIN_CUTOFF).values

X_train_raw, X_val_raw = X_all[mask], X_all[~mask]
y_train, y_val         = y_all[mask], y_all[~mask]
print(f"  Train : {mask.sum():,}  |  Val : {(~mask).sum():,}")

imp     = SimpleImputer(strategy="median")
X_train = imp.fit_transform(X_train_raw)
X_val   = imp.transform(X_val_raw)

# ═══════════════════════════════════════════════════════════════
# 3. XGBoost BRANCH  (GPU)
# ═══════════════════════════════════════════════════════════════
print("\n[1/3] Training XGBoost branch …")

xgb_model = xgb.XGBRegressor(
    n_estimators          = 800,
    learning_rate         = 0.05,
    max_depth             = 7,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    tree_method           = "hist",
    device                = xgb_device,
    n_jobs                = -1,
    random_state          = 42,
    early_stopping_rounds = 50,
    eval_metric           = "mae",
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

xgb_train_pred = xgb_model.predict(X_train).astype(np.float32)
xgb_val_pred   = xgb_model.predict(X_val).astype(np.float32)
mae_xgb = mean_absolute_error(y_val, xgb_val_pred)
r2_xgb  = r2_score(y_val, xgb_val_pred)
print(f"\n  XGB  → MAE: ₹{mae_xgb:.2f}  |  R²: {r2_xgb:.4f}")

# ═══════════════════════════════════════════════════════════════
# 4. LSTM BRANCH  (CPU)
# ═══════════════════════════════════════════════════════════════
print("\n[2/3] Training LSTM branch (CPU) …")

feat_idx   = {f: i for i, f in enumerate(available_price_feats)}
seq_idx    = [feat_idx[f] for f in SEQ_FEATS if f in feat_idx]

seq_scaler = StandardScaler()
seq_train  = seq_scaler.fit_transform(X_train[:, seq_idx])
seq_val    = seq_scaler.transform(X_val[:, seq_idx])

y_scaler  = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_s   = y_scaler.transform(y_val.reshape(-1, 1)).ravel()


def make_tensors(seq, y_scaled):
    X_t = torch.tensor(seq, dtype=torch.float32).unsqueeze(1)
    y_t = torch.tensor(y_scaled, dtype=torch.float32).unsqueeze(1)
    return TensorDataset(X_t, y_t)


BATCH    = 4096
train_dl = DataLoader(make_tensors(seq_train, y_train_s), batch_size=BATCH, shuffle=True,  num_workers=0)
val_dl   = DataLoader(make_tensors(seq_val,   y_val_s),   batch_size=BATCH, shuffle=False, num_workers=0)


class PriceLSTM(nn.Module):
    def __init__(self, input_size, hidden=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0.0,
            batch_first = True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


lstm_model = PriceLSTM(input_size=len(seq_idx)).to(lstm_device)
optimizer  = torch.optim.AdamW(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion  = nn.HuberLoss(delta=1.0)

EPOCHS, best_val, best_state = 25, float("inf"), None

for epoch in range(1, EPOCHS + 1):
    lstm_model.train()
    tr_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(lstm_device), yb.to(lstm_device)
        optimizer.zero_grad()
        loss = criterion(lstm_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item() * len(xb)
    tr_loss /= len(train_dl.dataset)

    lstm_model.eval()
    vl_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(lstm_device), yb.to(lstm_device)
            vl_loss += criterion(lstm_model(xb), yb).item() * len(xb)
    vl_loss /= len(val_dl.dataset)
    scheduler.step(vl_loss)

    if vl_loss < best_val:
        best_val   = vl_loss
        best_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{EPOCHS} | train={tr_loss:.4f} | val={vl_loss:.4f}")

lstm_model.load_state_dict(best_state)
lstm_model.eval()


def lstm_predict(seq_np):
    preds = []
    ds = torch.tensor(seq_np, dtype=torch.float32).unsqueeze(1)
    for i in range(0, len(ds), BATCH):
        with torch.no_grad():
            preds.append(lstm_model(ds[i:i + BATCH].to(lstm_device)).cpu().numpy().ravel())
    return np.concatenate(preds)


lstm_train_pred = y_scaler.inverse_transform(lstm_predict(seq_train).reshape(-1, 1)).ravel()
lstm_val_pred   = y_scaler.inverse_transform(lstm_predict(seq_val).reshape(-1, 1)).ravel()

mae_lstm = mean_absolute_error(y_val, lstm_val_pred)
r2_lstm  = r2_score(y_val, lstm_val_pred)
print(f"\n  LSTM → MAE: ₹{mae_lstm:.2f}  |  R²: {r2_lstm:.4f}")

# ═══════════════════════════════════════════════════════════════
# 5. META-LEARNER
# ═══════════════════════════════════════════════════════════════
print("\n[3/3] Fitting Ridge meta-stacker …")

meta_model = Ridge(alpha=1.0)
meta_model.fit(np.column_stack([xgb_train_pred, lstm_train_pred]), y_train)
final_preds = meta_model.predict(np.column_stack([xgb_val_pred, lstm_val_pred]))

print(f"  XGB weight  : {meta_model.coef_[0]:.4f}")
print(f"  LSTM weight : {meta_model.coef_[1]:.4f}")

# ═══════════════════════════════════════════════════════════════
# 6. RESULTS
# ═══════════════════════════════════════════════════════════════
mae  = mean_absolute_error(y_val, final_preds)
rmse = mean_squared_error(y_val, final_preds) ** 0.5
r2   = r2_score(y_val, final_preds)
mape = np.mean(np.abs((y_val - final_preds) / np.clip(y_val, 1, None))) * 100

print("\n" + "=" * 55)
print("  XGB + LSTM HYBRID — FINAL RESULTS")
print("=" * 55)
print(f"  XGBoost alone  → MAE ₹{mae_xgb:.2f}  R² {r2_xgb:.4f}")
print(f"  LSTM alone     → MAE ₹{mae_lstm:.2f}  R² {r2_lstm:.4f}")
print(f"  Hybrid stacked → MAE ₹{mae:.2f}  R² {r2:.4f}")
print(f"  RMSE           : ₹{rmse:.2f}")
print(f"  MAPE           : {mape:.2f}%")
print("=" * 55)

# ═══════════════════════════════════════════════════════════════
# 7. SAVE
# ═══════════════════════════════════════════════════════════════
joblib.dump(xgb_model,  OUTPUT_DIR / "xgb_price_model.pkl")
joblib.dump(imp,        OUTPUT_DIR / "price_imputer.pkl")
joblib.dump(seq_scaler, OUTPUT_DIR / "lstm_seq_scaler.pkl")
joblib.dump(y_scaler,   OUTPUT_DIR / "lstm_y_scaler.pkl")
joblib.dump(meta_model, OUTPUT_DIR / "meta_stacker.pkl")
torch.save(best_state,  OUTPUT_DIR / "lstm_price_model.pt")
print(f"\n  Artefacts saved → {OUTPUT_DIR}")

# ═══════════════════════════════════════════════════════════════
# 8. INFERENCE HELPER
# ═══════════════════════════════════════════════════════════════
def predict_price(feature_dict: dict) -> float:
    """Single-row inference. Keys must match available_price_feats."""
    row    = np.array([[feature_dict.get(f, np.nan) for f in available_price_feats]], dtype=np.float32)
    row    = imp.transform(row)

    xgb_p  = xgb_model.predict(row)[0]

    seq_row = seq_scaler.transform(row[:, seq_idx])
    seq_t   = torch.tensor(seq_row, dtype=torch.float32).unsqueeze(1).to(lstm_device)
    with torch.no_grad():
        lstm_p = y_scaler.inverse_transform([[lstm_model(seq_t).cpu().item()]])[0][0]

    return float(meta_model.predict([[xgb_p, lstm_p]])[0])

XGBoost device : cuda
LSTM device    : cpu (forced)

Loading merged-engineered parquet …
Loaded 13,243,611 rows  |  1 file(s)
  Sampled to 5M rows
  Train : 4,172,116  |  Val : 827,884

[1/3] Training XGBoost branch …
[0]	validation_0-mae:1580.30608
[100]	validation_0-mae:668.43398
[200]	validation_0-mae:653.87786
[300]	validation_0-mae:651.70706
[400]	validation_0-mae:650.99996
[500]	validation_0-mae:649.71612
[600]	validation_0-mae:648.70605
[700]	validation_0-mae:648.22548
[799]	validation_0-mae:648.00921

  XGB  → MAE: ₹647.99  |  R²: 0.7318

[2/3] Training LSTM branch (CPU) …
  Epoch   1/25 | train=0.0710 | val=0.1529
  Epoch   5/25 | train=0.0633 | val=0.1504
  Epoch  10/25 | train=0.0631 | val=0.1495
  Epoch  15/25 | train=0.0626 | val=0.1494
  Epoch  20/25 | train=0.0625 | val=0.1498
  Epoch  25/25 | train=0.0623 | val=0.1492

  LSTM → MAE: ₹660.33  |  R²: 0.7862

[3/3] Fitting Ridge meta-stacker …
  XGB weight  : 1.1605
  LSTM weight : -0.1594

  XGB + LSTM HYBRID — FINAL RESU